# 02 · Results

Shows the output of one pipeline run. Run the pipeline first:

```bash
python scripts/run_pipeline.py --config configs/sample.yaml      # fake sample data
python scripts/run_pipeline.py --config configs/nist_x4.yaml     # real NIST data
```

Then set `RUN` below to the output folder.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))
from src import plots

RUN = REPO / "results/sample"          # or REPO / "results/nist_x4"
info = json.loads((RUN / "run_info.json").read_text())
print("methods:", info["methods"])
print("segments:", info["n_segments"], "| injected faults:", info["n_injected_faults"])
print("XCT pore threshold:", info["xct_threshold"])

## 1. Tables

In [ ]:
display(Markdown((RUN / "results.md").read_text()))

## 2. Precision-recall curves (injected faults)

A curve closer to the top right is better. The dashed line is what random guessing would give.

In [ ]:
display(Image(str(RUN / "figures/pr_curves.png"), width=650))

## 3. Which faults are caught?

In [ ]:
display(Image(str(RUN / "figures/fault_types.png"), width=650))
display(Image(str(RUN / "figures/signal_examples.png"), width=950))

## 4. Where do false rejections come from?

In [ ]:
display(Image(str(RUN / "figures/frr_by_region.png"), width=560))

## 5. Traceability: every alarm on the clean test data

Each row is one flagged segment: where it is, which methods flagged it, and what XCT says about that spot. `alarm_ratio`: 0 = typical normal segment, 1 = exactly at the alarm level.

In [ ]:
trace = pd.read_csv(RUN / "traceability.csv")
print(len(trace), "flagged segments")
print("flagged by how many methods:", trace["n_methods"].value_counts().sort_index().to_dict())
if "xct_defect" in trace:
    print(f"share of flagged segments with an XCT pore: {trace['xct_defect'].mean():.1%}")
trace.head(15)

## 6. Look at any layer

Choose a method and a layer. Segments are drawn as the line the laser travelled; red rings are alarms, black crosses are XCT pores.

In [ ]:
scores = pd.read_parquet(RUN / "scores_test.parquet")
METHOD = "limits"                       # any of info["methods"] except limits_3sigma
part = int(scores["part"].iloc[0])
# the layer with the most alarms for this method
LAYER = int(scores[scores["part"] == part].groupby("layer")[f"flag_{METHOD}"].sum().idxmax())

centre, level = info["typical_normal_score"][METHOD], info["thresholds"][METHOD]
ratio = (scores[f"score_{METHOD}"] - centre) / (level - centre)
fig = plots.plot_layer_heatmap(scores, LAYER, part, ratio, scores[f"flag_{METHOD}"], METHOD)
display(fig)